# Whisper Audio Drive — Best Open Source Transcriber

Notebook ini dibuat untuk flow:

1. Load model **faster-whisper large-v3**
2. Mount Google Drive
3. Cek audio/video yang sudah ada di folder Drive
4. Transcribe langsung jadi:
   - `.txt` format timestamp + speaker seperti contoh
   - `.md`
   - `.srt`
   - `.vtt`
   - `.json`

Folder input default:

```txt
/content/drive/MyDrive/Whisper/AudioFiles
```

Folder output default:

```txt
/content/drive/MyDrive/Whisper/Transcripts
```

> Penting: aktifkan GPU di Colab: **Runtime → Change runtime type → T4 GPU / L4 GPU / A100**.


In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip -q install -U faster-whisper
!apt-get -qq update
!apt-get -qq install -y ffmpeg

print("✅ Dependencies installed.")


In [ ]:
# ============================================================
# CELL 2 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")
print("✅ Google Drive mounted.")


In [ ]:
# ============================================================
# CELL 3 — CONFIG
# ============================================================

import os
import json
import shutil
import subprocess
from pathlib import Path

# =========================
# FOLDER SETTING
# =========================

AUDIO_FOLDER = "/content/drive/MyDrive/Whisper/AudioFiles"
OUTPUT_FOLDER = "/content/drive/MyDrive/Whisper/Transcripts"

# =========================
# MODEL SETTING
# =========================
# Rekomendasi:
# - "large-v3" = akurasi open-source terbaik, lebih berat
# - "large-v3-turbo" = lebih cepat, tetap bagus
# - "medium" = lebih ringan, akurasi turun
# - "small" = cepat, tapi hasil bisa jelek

MODEL_NAME = "large-v3"

# Bahasa:
# - "id" untuk Bahasa Indonesia
# - None untuk auto-detect, tapi untuk konten Indonesia lebih stabil pakai "id"
LANGUAGE = "id"

# Speaker default karena faster-whisper biasa belum punya diarization speaker otomatis
DEFAULT_SPEAKER = "Speaker 1"

# Kalau True, file yang output .txt-nya sudah ada akan dilewati
SKIP_EXISTING = False

# Kalau True, audio akan dinormalisasi dulu ke WAV mono 16kHz.
# Ini sering membantu untuk audio video/HP/recording yang volumenya tidak rata.
PREPROCESS_AUDIO = True

# Ekstensi yang akan diproses
SUPPORTED_EXTENSIONS = [
    ".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".wma",
    ".mp4", ".mov", ".mkv", ".webm"
]

# Prompt konteks supaya nama/istilah tidak ngawur
INITIAL_PROMPT = """
Transkrip Bahasa Indonesia yang rapi dan akurat.
Pertahankan nama orang, tempat, lembaga, dan istilah penting dengan benar.
Konteks umum: berita, wawancara, liputan, pendidikan, politik daerah, Semarang, Jawa Tengah, DPRD, Komisi E, Golkar, Dipa Yustiapasa, SPMB, SMA Negeri 3 Semarang.
Jangan menerjemahkan istilah. Jangan menambahkan informasi yang tidak ada di audio.
"""

os.makedirs(AUDIO_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("✅ Config loaded.")
print("Input folder :", AUDIO_FOLDER)
print("Output folder:", OUTPUT_FOLDER)
print("Model        :", MODEL_NAME)
print("Language     :", LANGUAGE)


In [ ]:
# ============================================================
# CELL 4 — LOAD MODEL
# ============================================================

import torch
from faster_whisper import WhisperModel

if torch.cuda.is_available():
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"
    print("✅ GPU detected:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"
    print("⚠️ GPU tidak terdeteksi. Akan jalan di CPU dan bisa sangat lambat.")

print(f"Loading model: {MODEL_NAME} | device={DEVICE} | compute_type={COMPUTE_TYPE}")

try:
    model = WhisperModel(
        MODEL_NAME,
        device=DEVICE,
        compute_type=COMPUTE_TYPE
    )
except Exception as e:
    print("⚠️ Gagal load dengan compute_type:", COMPUTE_TYPE)
    print("Error:", e)
    print("Mencoba fallback ke int8_float16 untuk GPU atau int8 untuk CPU...")

    COMPUTE_TYPE = "int8_float16" if DEVICE == "cuda" else "int8"
    model = WhisperModel(
        MODEL_NAME,
        device=DEVICE,
        compute_type=COMPUTE_TYPE
    )

print("✅ Model loaded successfully.")


In [ ]:
# ============================================================
# CELL 5 — HELPER FUNCTIONS
# ============================================================

def format_timestamp_comma(seconds: float) -> str:
    """
    Format timestamp:
    00:00:00,100
    """
    if seconds is None:
        seconds = 0.0

    total_ms = int(round(float(seconds) * 1000))
    hours = total_ms // 3_600_000
    minutes = (total_ms % 3_600_000) // 60_000
    secs = (total_ms % 60_000) // 1000
    millis = total_ms % 1000

    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


def format_timestamp_dot(seconds: float) -> str:
    """
    Format timestamp untuk VTT:
    00:00:00.100
    """
    return format_timestamp_comma(seconds).replace(",", ".")


def clean_text(text: str) -> str:
    """
    Membersihkan spasi berlebih tanpa mengubah isi kalimat.
    """
    if not text:
        return ""
    return " ".join(text.strip().split())


def preprocess_to_wav(input_path: Path, temp_dir: str = "/content/whisper_preprocessed") -> Path:
    """
    Convert audio/video ke WAV mono 16kHz dengan loudness normalization.
    Ini membantu untuk file dari video, HP, atau audio yang volumenya tidak rata.
    """
    os.makedirs(temp_dir, exist_ok=True)

    output_path = Path(temp_dir) / f"{input_path.stem}_preprocessed.wav"

    command = [
        "ffmpeg",
        "-y",
        "-i", str(input_path),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-af", "loudnorm=I=-16:TP=-1.5:LRA=11",
        str(output_path)
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        print("⚠️ Preprocess gagal. Akan pakai file asli.")
        print(result.stderr[-1000:])
        return input_path

    return output_path


def save_txt_like_example(segments, output_path, default_speaker=DEFAULT_SPEAKER):
    """
    Format output txt:

    00:00:00,100 --> 00:00:33,940 [Speaker 1]
    Isi transkrip...

    00:00:33,940 --> 00:03:27,580 [Speaker 1]
    Isi transkrip berikutnya...
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for seg in segments:
            start = format_timestamp_comma(seg["start"])
            end = format_timestamp_comma(seg["end"])
            text = clean_text(seg["text"])

            if not text:
                continue

            speaker = seg.get("speaker", default_speaker)

            f.write(f"{start} --> {end} [{speaker}]\n")
            f.write(f"{text}\n\n")


def save_markdown_transcript(segments, output_path, title="Transkrip Audio", default_speaker=DEFAULT_SPEAKER):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")

        for seg in segments:
            start = format_timestamp_comma(seg["start"])
            end = format_timestamp_comma(seg["end"])
            text = clean_text(seg["text"])

            if not text:
                continue

            speaker = seg.get("speaker", default_speaker)

            f.write(f"### {start} → {end} [{speaker}]\n\n")
            f.write(f"{text}\n\n")


def save_srt(segments, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        index = 1

        for seg in segments:
            start = format_timestamp_comma(seg["start"])
            end = format_timestamp_comma(seg["end"])
            text = clean_text(seg["text"])

            if not text:
                continue

            f.write(f"{index}\n")
            f.write(f"{start} --> {end}\n")
            f.write(f"{text}\n\n")

            index += 1


def save_vtt(segments, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("WEBVTT\n\n")

        for seg in segments:
            start = format_timestamp_dot(seg["start"])
            end = format_timestamp_dot(seg["end"])
            text = clean_text(seg["text"])

            if not text:
                continue

            f.write(f"{start} --> {end}\n")
            f.write(f"{text}\n\n")


def save_json(segments, info, output_path):
    data = {
        "model": MODEL_NAME,
        "language": getattr(info, "language", None),
        "language_probability": getattr(info, "language_probability", None),
        "duration": getattr(info, "duration", None),
        "segments": segments
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


print("✅ Helper functions ready.")


In [ ]:
# ============================================================
# CELL 6 — FIND AUDIO / VIDEO FILES
# ============================================================

def find_audio_files(folder_path: str):
    folder = Path(folder_path)

    files = []
    for file_path in folder.rglob("*"):
        if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS:
            files.append(file_path)

    return sorted(files)


audio_files = find_audio_files(AUDIO_FOLDER)

print(f"Total file audio/video ditemukan: {len(audio_files)}")

if len(audio_files) == 0:
    print("")
    print("⚠️ Belum ada audio/video.")
    print("Upload file ke folder:")
    print(AUDIO_FOLDER)
else:
    for i, file in enumerate(audio_files, start=1):
        print(f"{i}. {file}")


In [ ]:
# ============================================================
# CELL 7 — TRANSCRIBE AUDIO TO TXT / MD / SRT / VTT / JSON
# ============================================================

def transcribe_file(audio_path: Path):
    print("=" * 90)
    print(f"🎧 Processing: {audio_path.name}")

    base_name = audio_path.stem

    txt_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.txt")
    md_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.md")
    srt_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.srt")
    vtt_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.vtt")
    json_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.json")

    if SKIP_EXISTING and os.path.exists(txt_path):
        print(f"⏭️ Skip karena output sudah ada: {txt_path}")
        return {
            "audio": str(audio_path),
            "txt": txt_path,
            "md": md_path,
            "srt": srt_path,
            "vtt": vtt_path,
            "json": json_path,
            "skipped": True
        }

    input_for_transcribe = audio_path

    if PREPROCESS_AUDIO:
        print("🔧 Preprocessing audio/video ke WAV mono 16kHz...")
        input_for_transcribe = preprocess_to_wav(audio_path)
        print("Preprocessed file:", input_for_transcribe)

    print("📝 Transcribing...")

    segments_generator, info = model.transcribe(
        str(input_for_transcribe),

        # Bahasa Indonesia
        language=LANGUAGE,
        task="transcribe",

        # Akurasi
        beam_size=5,
        best_of=5,
        temperature=0.0,

        # Stabil untuk audio panjang
        vad_filter=True,
        vad_parameters={
            "min_silence_duration_ms": 500
        },

        # Konteks antar segment tetap nyambung
        condition_on_previous_text=True,

        # Prompt khusus konteks
        initial_prompt=INITIAL_PROMPT,

        # False = lebih cepat dan cukup untuk output segment
        # True = detail kata per kata, tapi lebih berat
        word_timestamps=False
    )

    segments = []

    for segment in segments_generator:
        text = segment.text.strip()

        item = {
            "id": segment.id,
            "start": float(segment.start),
            "end": float(segment.end),
            "speaker": DEFAULT_SPEAKER,
            "text": text
        }

        segments.append(item)

        print(
            f"{format_timestamp_comma(segment.start)} --> {format_timestamp_comma(segment.end)} "
            f"[{DEFAULT_SPEAKER}] {text}"
        )

    save_txt_like_example(segments, txt_path, default_speaker=DEFAULT_SPEAKER)
    save_markdown_transcript(segments, md_path, title=base_name, default_speaker=DEFAULT_SPEAKER)
    save_srt(segments, srt_path)
    save_vtt(segments, vtt_path)
    save_json(segments, info, json_path)

    print("")
    print("✅ Saved outputs:")
    print("TXT :", txt_path)
    print("MD  :", md_path)
    print("SRT :", srt_path)
    print("VTT :", vtt_path)
    print("JSON:", json_path)

    return {
        "audio": str(audio_path),
        "txt": txt_path,
        "md": md_path,
        "srt": srt_path,
        "vtt": vtt_path,
        "json": json_path,
        "skipped": False
    }


results = []

if len(audio_files) == 0:
    print("⚠️ Tidak ada file yang diproses.")
    print("Upload audio/video ke:")
    print(AUDIO_FOLDER)
else:
    for audio_file in audio_files:
        try:
            result = transcribe_file(audio_file)
            results.append(result)
        except Exception as e:
            print("❌ Error saat memproses:", audio_file)
            print(e)

print("=" * 90)
print("✅ Semua proses selesai.")
print(f"Total output: {len(results)} file diproses/dicek.")


In [ ]:
# ============================================================
# CELL 8 — SHOW RESULT SUMMARY
# ============================================================

if "results" in globals() and len(results) > 0:
    print("📁 Ringkasan output:")
    for item in results:
        print("-" * 90)
        print("Audio:", item["audio"])
        print("TXT  :", item["txt"])
        print("MD   :", item["md"])
        print("SRT  :", item["srt"])
        print("VTT  :", item["vtt"])
        print("JSON :", item["json"])
else:
    print("Belum ada results. Jalankan CELL 7 dulu.")


## Contoh output `.txt`

```txt
00:00:00,100 --> 00:00:33,940 [Speaker 1]
[musik intro bersemangat]

00:00:33,940 --> 00:03:27,580 [Speaker 1]
Setiap tahun, ribuan siswa bermimpi bisa belajar di sekolah terbaik...
```

## Catatan speaker

Notebook ini memakai `faster-whisper`, jadi belum ada speaker diarization otomatis.  
Label speaker akan default menjadi `[Speaker 1]`.

Kalau butuh otomatis membedakan `[Speaker 0]`, `[Speaker 1]`, dan seterusnya, perlu versi lanjutan memakai:

```txt
WhisperX + pyannote diarization + HuggingFace token
```
